In [1]:
def extract_event_number(filename):
    """
    Extract event number from filenames like:
    res_23_1993_1_Ens07_binary_10cm.tif -> 1
    res_23_1993_123_Ens07_binary_10cm.tif -> 123
    """
    match = re.search(r'_(\d+)_Ens\d+_(?:binary|filtered)_', filename)
    if match:
        return int(match.group(1))
    return None


def iter_event_tifs(ha_num, data_kind, ensembles=None, thresholds=None):
    """
    Yield metadata for every event tif in:
    tifs/EnsXX_<HA_NUM>/<data_kind>/10cm/*.tif
    tifs/EnsXX_<HA_NUM>/<data_kind>/30cm/*.tif
    """
    if data_kind not in {"binary", "filtered"}:
        raise ValueError(f"Unsupported data_kind={data_kind}")

    ensembles = ensembles or ENSEMBLE_MEMBERS
    thresholds = thresholds or THRESHOLDS
    ha_num = str(ha_num)

    for ens in ensembles:
        ens_name = f"Ens{ens}_{ha_num}"

        for thr in thresholds:
            tif_glob = os.path.join(TIFS_DIR, ens_name, data_kind, thr, "*.tif")
            tif_paths = sorted(glob.glob(tif_glob))

            if not tif_paths:
                print(f"[WARN] No files found: {os.path.dirname(tif_glob)}")
                continue

            for tif_path in tif_paths:
                event_num = extract_event_number(os.path.basename(tif_path))
                if event_num is None:
                    print(f"[WARN] Could not extract event number from: {os.path.basename(tif_path)}")
                    continue
                
                yield {
                    "ha_num": ha_num,
                    "ensemble": ens_name,
                    "threshold": thr,
                    "data_kind": data_kind,
                    "path": tif_path,
                    "event_num": event_num,
                }


def process_single_event_area(binary_file, x5, y5, nx, ny, qa_out_tif=None):
    """
    Process a single binary flood tif and return flooded area per 5km grid cell.
    Maps 30m pixels directly to the original 5km grid using coordinates.
    """
    # Open flood raster
    flood = rxr.open_rasterio(
        binary_file,
        chunks={"x": 2000, "y": 2000}
    ).squeeze()

    if flood.rio.crs is None or flood.rio.crs.to_string() != "EPSG:27700":
        flood = flood.rio.reproject("EPSG:27700")

    # Identify valid pixels
    valid = flood.notnull()

    # Get flood pixel coordinates and values
    flood_x = flood.x.values
    flood_y = flood.y.values
    
    # Create meshgrid of flood coordinates
    xx, yy = np.meshgrid(flood_x, flood_y)
    
    # Flatten coordinates and values
    x_flat = xx.ravel()
    y_flat = yy.ravel()
    flood_flat = flood.values.ravel()
    valid_flat = valid.values.ravel()
    
    # Only keep valid, flooded pixels
    mask = valid_flat & (flood_flat > 0)
    x_flood = x_flat[mask]
    y_flood = y_flat[mask]
    
    if x_flood.size == 0:
        return np.zeros((ny, nx), dtype=np.float32)

    # Map flood-pixel centers to 5km cell indices using cell edges.
    x5_arr = np.asarray(x5.values)
    y5_arr = np.asarray(y5.values)

    dx5 = float(abs(x5_arr[1] - x5_arr[0]))
    dy5 = float(abs(y5_arr[1] - y5_arr[0]))

    if x5_arr[1] > x5_arr[0]:
        xmin_edge = float(np.min(x5_arr) - dx5 / 2.0)
        ix = np.floor((x_flood - xmin_edge) / dx5).astype(np.int64)
    else:
        xmax_edge = float(np.max(x5_arr) + dx5 / 2.0)
        ix = np.floor((xmax_edge - x_flood) / dx5).astype(np.int64)

    if y5_arr[1] > y5_arr[0]:
        ymin_edge = float(np.min(y5_arr) - dy5 / 2.0)
        iy = np.floor((y_flood - ymin_edge) / dy5).astype(np.int64)
    else:
        ymax_edge = float(np.max(y5_arr) + dy5 / 2.0)
        iy = np.floor((ymax_edge - y_flood) / dy5).astype(np.int64)

    in_grid = (ix >= 0) & (ix < nx) & (iy >= 0) & (iy < ny)
    ix = ix[in_grid]
    iy = iy[in_grid]

    if ix.size == 0:
        return np.zeros((ny, nx), dtype=np.float32)
    
    # Convert 2D indices to 1D linear indices
    linear_idx = iy * nx + ix

    if qa_out_tif is not None:
        # Build QA raster in original 30m grid:
        # value = 5km linear cell index for flooded pixels, -1 elsewhere.
        flooded_flat_idx = np.where(mask)[0]
        flooded_flat_idx_in_grid = flooded_flat_idx[in_grid]
        qa_flat = np.full(flood_flat.shape, -1, dtype=np.int32)
        qa_flat[flooded_flat_idx_in_grid] = linear_idx.astype(np.int32)
        qa_arr = qa_flat.reshape(flood.shape)

        qa_da = xr.DataArray(qa_arr, coords=flood.coords, dims=flood.dims)
        qa_da = qa_da.rio.write_crs(flood.rio.crs)
        qa_da.rio.write_transform(flood.rio.transform(), inplace=True)

        os.makedirs(os.path.dirname(qa_out_tif), exist_ok=True)
        qa_da.rio.to_raster(qa_out_tif)
    
    # Count flooded pixels per grid cell
    counts_flat = np.bincount(linear_idx, minlength=nx * ny)
    counts = counts_flat.reshape(ny, nx)
    
    # Convert flooded-pixel counts to total flooded area using actual raster resolution.
    xres, yres = flood.rio.resolution()
    pixel_area_km2 = (abs(xres) * abs(yres)) / 1e6
    flood_area = counts.astype(np.float32) * pixel_area_km2
    
    return flood_area


def process_single_event_volume(depth_file, x5, y5, nx, ny):
    """
    Process a single depth flood tif and return flooded volume per 5km grid cell.
    Volume is computed as sum(depth_m * pixel_area_m2) over pixels within each 5km cell.
    """
    depth = rxr.open_rasterio(
        depth_file,
        chunks={"x": 2000, "y": 2000}
    ).squeeze()

    if depth.rio.crs is None or depth.rio.crs.to_string() != "EPSG:27700":
        depth = depth.rio.reproject("EPSG:27700")

    valid = depth.notnull()

    depth_x = depth.x.values
    depth_y = depth.y.values
    xx, yy = np.meshgrid(depth_x, depth_y)

    x_flat = xx.ravel()
    y_flat = yy.ravel()
    depth_flat = depth.values.ravel()
    valid_flat = valid.values.ravel()

    # Only include valid, positive depths in the volume sum.
    mask = valid_flat & (depth_flat > 0)
    x_depth = x_flat[mask]
    y_depth = y_flat[mask]
    depth_vals = depth_flat[mask].astype(np.float64)

    if x_depth.size == 0:
        return np.zeros((ny, nx), dtype=np.float32)

    x5_arr = np.asarray(x5.values)
    y5_arr = np.asarray(y5.values)

    dx5 = float(abs(x5_arr[1] - x5_arr[0]))
    dy5 = float(abs(y5_arr[1] - y5_arr[0]))

    if x5_arr[1] > x5_arr[0]:
        xmin_edge = float(np.min(x5_arr) - dx5 / 2.0)
        ix = np.floor((x_depth - xmin_edge) / dx5).astype(np.int64)
    else:
        xmax_edge = float(np.max(x5_arr) + dx5 / 2.0)
        ix = np.floor((xmax_edge - x_depth) / dx5).astype(np.int64)

    if y5_arr[1] > y5_arr[0]:
        ymin_edge = float(np.min(y5_arr) - dy5 / 2.0)
        iy = np.floor((y_depth - ymin_edge) / dy5).astype(np.int64)
    else:
        ymax_edge = float(np.max(y5_arr) + dy5 / 2.0)
        iy = np.floor((ymax_edge - y_depth) / dy5).astype(np.int64)

    in_grid = (ix >= 0) & (ix < nx) & (iy >= 0) & (iy < ny)
    ix = ix[in_grid]
    iy = iy[in_grid]
    depth_vals = depth_vals[in_grid]

    if ix.size == 0:
        return np.zeros((ny, nx), dtype=np.float32)

    linear_idx = iy * nx + ix

    xres, yres = depth.rio.resolution()
    pixel_area_m2 = abs(xres) * abs(yres)

    # Depth rasters are in meters, so depth[m] * area[m2] -> volume[m3].
    contrib_m3 = depth_vals * pixel_area_m2
    volume_flat = np.bincount(linear_idx, weights=contrib_m3, minlength=nx * ny)
    flood_volume = volume_flat.reshape(ny, nx).astype(np.float32)

    return flood_volume

In [2]:
import os
import glob
import re
import argparse
import xarray as xr
import rioxarray as rxr
import numpy as np
from rasterio.transform import from_bounds

# -----------------------------
# CONFIG
# -----------------------------
HA_NUM = 12

ROOT_DIR = "/scratch/hydro5/users/la17355/FUTURE-FLOOD/Results/Pluvial/v4"
# OUTPUT_DIR = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/5km_total/Catchment_{HA_NUM}"

TIFS_DIR = os.path.join(ROOT_DIR, "tifs")

# Your 12 ensembles
ENSEMBLE_MEMBERS = ["01", "04", "05", "06", "07", "08", "09", "10", "11", "12", "13", "15"]

# Only the two thresholds you care about
THRESHOLDS = ["10cm", "30cm"]

# QA output: write 30m geotiffs showing assigned 5km cell id per flooded pixel.
WRITE_QA_GRID_INDEX_TIF = False
QA_MAX_EVENTS_PER_THRESHOLD = 1

# Input grid
GRID_5KM_FILE = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/UKCP_rainfall/5km/Ens_01/bc_pr_rcp85_land-cpm_uk_5km_01_1hr_19901201-19911130.nc"
# GRID_5KM_FILE = "/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/01/bc_pr_rcp85_land-cpm_uk_5km_01_1hr_20801101-20801130.nc"

In [6]:
files = os.listdir("/scratch/hydro5/users/la17355/FUTURE-FLOOD/Results/Pluvial/v4/tifs/")

# Extract everything after 'Ens15_'
catchment_numbers = set()
for f in files:
    match = re.search(r'Ens15_(.+)', f)
    if match:
        catchment_numbers.add(match.group(1))

# -----------------------------
# FILTER TO ONLY INCOMPLETE CATCHMENTS
# -----------------------------
def all_outputs_exist(ha_num):
    out_dir_base = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/5km_total/Catchment_{ha_num}"
    for ens in ENSEMBLE_MEMBERS:
        for thr in THRESHOLDS:
            out_dir = os.path.join(out_dir_base, f"Ens{ens}_{ha_num}", thr)
            for kind in ("area", "volume"):
                fname = f"flooded_{kind}_5km_total_Ens{ens}_{ha_num}_{thr}.nc"
                if not os.path.exists(os.path.join(out_dir, fname)):
                    return False
    return True

catchments_to_skip = {c for c in catchment_numbers if all_outputs_exist(c)}
catchments_to_run = catchment_numbers - catchments_to_skip

print(f"{len(catchments_to_skip)} catchments already complete, skipping.")
print(f"{len(catchments_to_run)} catchments to process: {sorted(catchments_to_run)}")

25 catchments already complete, skipping.
62 catchments to process: ['105', '107', '11', '14', '16', '17', '18', '19', '2', '22', '27_a', '27_b', '27_c', '28_a', '28_b', '32', '33_b', '34', '35', '37', '38', '4', '41', '42', '43', '45', '46', '48', '49', '5', '51', '54_a', '54_b', '54_c', '54_d', '58', '6', '61', '65', '66', '67', '69', '7', '70', '71', '74', '75', '76', '78', '79', '80', '83', '85', '87', '88', '89', '90', '91', '92', '93', '94', '95']


In [ ]:
for HA_NUM in catchments_to_run:
    print(f"Running for {HA_NUM}")
    OUT_DIR = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/5km_total/Catchment_{HA_NUM}"
    print(f"Outputs to be stored in {OUT_DIR}")
    
    # -----------------------------
    # OPEN 5km GRID
    # -----------------------------
    print(f"Loading 5km grid from {GRID_5KM_FILE}...")
    ds = xr.open_dataset(GRID_5KM_FILE)

    x5 = ds["projection_x_coordinate"]
    y5 = ds["projection_y_coordinate"]

    nx = x5.size
    ny = y5.size

    dx = float(abs(x5[1] - x5[0]))
    dy = float(abs(y5[1] - y5[0]))

    xmin = float(x5.min() - dx/2)
    xmax = float(x5.max() + dx/2)
    ymin = float(y5.min() - dy/2)
    ymax = float(y5.max() + dy/2)

    # -----------------------------
    # CREATE GRID-ID RASTER
    # -----------------------------
    grid_ids = np.arange(nx * ny).reshape(ny, nx)

    # Use standard y, x dimension names for rioxarray compatibility
    grid_da = xr.DataArray(
        grid_ids,
        coords={"y": y5.values, "x": x5.values},
        dims=("y", "x")
    )

    grid_da = grid_da.rio.write_crs("EPSG:27700")

    transform = from_bounds(xmin, ymin, xmax, ymax, nx, ny)
    grid_da.rio.write_transform(transform, inplace=True)

    # Export a GeoTIFF copy of the 5km grid IDs for visual QA.
    # grid_out_dir = os.path.join(OUT_DIR, "grid")
    # os.makedirs(grid_out_dir, exist_ok=True)
    # grid_tif_path = os.path.join(grid_out_dir, f"grid_id_5km_from_input_{HA_NUM}.tif")
    # print(f"Saving 5km grid GeoTIFF to {grid_tif_path}...")
    # grid_da.astype("int32").rio.to_raster(grid_tif_path)

    # -----------------------------
    # COLLECT ALL EVENT FILES
    # -----------------------------
    print(f"Scanning for binary tif files for HA_NUM={HA_NUM}...")
    binary_event_files = list(iter_event_tifs(HA_NUM, data_kind="binary"))

    print(f"Scanning for filtered depth tif files for HA_NUM={HA_NUM}...")
    filtered_event_files = list(iter_event_tifs(HA_NUM, data_kind="filtered"))

    if not binary_event_files:
        raise ValueError(f"No binary tif files found for HA_NUM={HA_NUM}")

    if not filtered_event_files:
        raise ValueError(f"No filtered depth tif files found for HA_NUM={HA_NUM}")

    print(f"Found {len(binary_event_files)} binary event files")
    print(f"Found {len(filtered_event_files)} filtered depth event files")

    # -----------------------------
    # GROUP BY ENSEMBLE
    # -----------------------------
    from collections import defaultdict
    binary_by_ensemble = defaultdict(list)
    filtered_lookup = {}

    for info in binary_event_files:
        binary_by_ensemble[info['ensemble']].append(info)

    for info in filtered_event_files:
        key = (info["ensemble"], info["threshold"], info["event_num"])
        filtered_lookup[key] = info

    print(f"Found {len(binary_by_ensemble)} ensemble members")

    # -----------------------------
    # PROCESS EACH ENSEMBLE + THRESHOLD SEPARATELY
    # -----------------------------
    for ens_name in sorted(binary_by_ensemble.keys()):
        ens_events = binary_by_ensemble[ens_name]
        ens_events.sort(key=lambda x: (x['threshold'], x['event_num']))

        print(f"\n{'='*60}")
        print(f"Processing {ens_name}: {len(ens_events)} total events")
        print(f"{'='*60}")

        for thr in THRESHOLDS:
            thr_events = [e for e in ens_events if e["threshold"] == thr]
            if not thr_events:
                print(f"[WARN] No events for {ens_name} threshold {thr}")
                continue

            print(f"\n[{ens_name} | {thr}] Processing {len(thr_events)} events")

            flood_areas = []
            flood_volumes = []
            for i, info in enumerate(thr_events):
                print(f"[{i+1}/{len(thr_events)}] Processing {os.path.basename(info['path'])} (event={info['event_num']})")

                key = (ens_name, thr, info["event_num"])
                if key not in filtered_lookup:
                    raise ValueError(
                        f"Missing filtered depth tif for {ens_name}, {thr}, event={info['event_num']}"
                    )

                filtered_info = filtered_lookup[key]

                qa_out_tif = None
                if WRITE_QA_GRID_INDEX_TIF and i < QA_MAX_EVENTS_PER_THRESHOLD:
                    print("Performing QA")
                    qa_dir = os.path.join(OUT_DIR, "qa", ens_name, thr)
                    qa_out_tif = os.path.join(
                        qa_dir,
                        f"qa_5km_cell_index_{ens_name}_{thr}_event_{info['event_num']:03d}.tif"
                    )
                    print(f"    Writing QA 30m->5km index raster: {qa_out_tif}")
                else:
                    print("Skippping QA")

                flood_area = process_single_event_area(
                    info['path'],
                    x5, y5, nx, ny,
                    qa_out_tif=qa_out_tif)

                flood_volume = process_single_event_volume(
                    filtered_info["path"],
                    x5, y5, nx, ny)

                total_km2 = float(np.sum(flood_area))
                total_m3 = float(np.sum(flood_volume))
                print(f"    Total flooded area (sum of 5km cells): {total_km2:.4f} km2")
                print(f"    Total flooded volume (sum of 5km cells): {total_m3:.2f} m3")
                flood_areas.append(flood_area)
                flood_volumes.append(flood_volume)

            flood_areas_stack = np.stack(flood_areas, axis=0)
            flood_volumes_stack = np.stack(flood_volumes, axis=0)
            event_nums = np.array([info['event_num'] for info in thr_events], dtype=np.int32)

            out_area = xr.Dataset(
                {
                    "flooded_area_5km_km2": (
                        ("event", "projection_y_coordinate", "projection_x_coordinate"),
                        flood_areas_stack
                    ),
                    "event_num": ("event", event_nums),
                },
                coords={
                    "event": event_nums,
                    "projection_x_coordinate": x5,
                    "projection_y_coordinate": y5
                }
            )

            out_volume = xr.Dataset(
                {
                    "flooded_volume_5km_m3": (
                        ("event", "projection_y_coordinate", "projection_x_coordinate"),
                        flood_volumes_stack
                    ),
                    "event_num": ("event", event_nums),
                },
                coords={
                    "event": event_nums,
                    "projection_x_coordinate": x5,
                    "projection_y_coordinate": y5
                }
            )

            for out_ds in (out_area, out_volume):
                out_ds["projection_x_coordinate"].attrs.update({
                    "standard_name": "projection_x_coordinate",
                    "long_name": "x coordinate of British National Grid projection",
                    "units": "m"
                })
                out_ds["projection_y_coordinate"].attrs.update({
                    "standard_name": "projection_y_coordinate",
                    "long_name": "y coordinate of British National Grid projection",
                    "units": "m"
                })
                out_ds["event_num"].attrs["long_name"] = "event number"
                out_ds["event"].attrs["long_name"] = "event number"

            out_area = out_area.rio.set_spatial_dims(
                x_dim="projection_x_coordinate",
                y_dim="projection_y_coordinate"
            )
            out_area.rio.write_transform(transform, inplace=True)
            out_area.rio.write_crs("EPSG:27700", inplace=True)
            out_area.rio.write_coordinate_system(inplace=True)

            out_volume = out_volume.rio.set_spatial_dims(
                x_dim="projection_x_coordinate",
                y_dim="projection_y_coordinate"
            )
            out_volume.rio.write_transform(transform, inplace=True)
            out_volume.rio.write_crs("EPSG:27700", inplace=True)
            out_volume.rio.write_coordinate_system(inplace=True)

            out_area["flooded_area_5km_km2"].attrs["long_name"] = "Total flooded area per 5km grid cell"
            out_area["flooded_area_5km_km2"].attrs["units"] = "km2"
            out_volume["flooded_volume_5km_m3"].attrs["long_name"] = "Total flooded volume per 5km grid cell"
            out_volume["flooded_volume_5km_m3"].attrs["units"] = "m3"

            out_dir = os.path.join(OUT_DIR, ens_name, thr)
            os.makedirs(out_dir, exist_ok=True)

            output_area_nc = os.path.join(out_dir, f"flooded_area_5km_total_{ens_name}_{thr}.nc")
            output_volume_nc = os.path.join(out_dir, f"flooded_volume_5km_total_{ens_name}_{thr}.nc")

            print(f"Saving area to {output_area_nc}...")
            out_area.to_netcdf(output_area_nc)
            print(f"Done! Saved {len(thr_events)} events to {output_area_nc}")

            print(f"Saving volume to {output_volume_nc}...")
            out_volume.to_netcdf(output_volume_nc)
            print(f"Done! Saved {len(thr_events)} events to {output_volume_nc}")

    print(f"\n{'='*60}")
    print(f"All {len(binary_by_ensemble)} ensemble members processed!")

Running for 105
Outputs to be stored in /scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/5km_total/Catchment_105
Loading 5km grid from /scratch/hydro4/users/la17355/FUTURE-FLOOD/UKCP_rainfall/5km/Ens_01/bc_pr_rcp85_land-cpm_uk_5km_01_1hr_19901201-19911130.nc...
Scanning for binary tif files for HA_NUM=105...
Scanning for filtered depth tif files for HA_NUM=105...
Found 6994 binary event files
Found 6994 filtered depth event files
Found 12 ensemble members

Processing Ens01_105: 308 total events

[Ens01_105 | 10cm] Processing 154 events
[1/154] Processing res_105_1992_1_Ens01_binary_10cm.tif (event=1)
Skippping QA
    Total flooded area (sum of 5km cells): 0.0252 km2
    Total flooded volume (sum of 5km cells): 5328.90 m3
[2/154] Processing res_105_1994_2_Ens01_binary_10cm.tif (event=2)
Skippping QA
    Total flooded area (sum of 5km cells): 0.8757 km2
    Total flooded volume (sum of 5km cells): 270560.69 m3
[3/154] Processing res_105_1999_3_Ens01_binary_10cm.tif (event=3)

    Total flooded area (sum of 5km cells): 0.5958 km2
    Total flooded volume (sum of 5km cells): 189672.30 m3
[41/154] Processing res_105_2046_41_Ens01_binary_10cm.tif (event=41)
Skippping QA
    Total flooded area (sum of 5km cells): 0.1206 km2
    Total flooded volume (sum of 5km cells): 34634.70 m3
[42/154] Processing res_105_2047_42_Ens01_binary_10cm.tif (event=42)
Skippping QA
    Total flooded area (sum of 5km cells): 0.0243 km2
    Total flooded volume (sum of 5km cells): 7558.20 m3
[43/154] Processing res_105_2047_43_Ens01_binary_10cm.tif (event=43)
Skippping QA
    Total flooded area (sum of 5km cells): 0.0153 km2
    Total flooded volume (sum of 5km cells): 4133.70 m3
[44/154] Processing res_105_2047_44_Ens01_binary_10cm.tif (event=44)
Skippping QA
    Total flooded area (sum of 5km cells): 0.1305 km2
    Total flooded volume (sum of 5km cells): 36963.00 m3
[45/154] Processing res_105_2047_45_Ens01_binary_10cm.tif (event=45)
Skippping QA
    Total flooded area (sum of 5km c

    Total flooded area (sum of 5km cells): 1.8423 km2
    Total flooded volume (sum of 5km cells): 599459.38 m3
[84/154] Processing res_105_2070_84_Ens01_binary_10cm.tif (event=84)
Skippping QA
    Total flooded area (sum of 5km cells): 2.4939 km2
    Total flooded volume (sum of 5km cells): 1261934.12 m3
[85/154] Processing res_105_2071_85_Ens01_binary_10cm.tif (event=85)
Skippping QA
    Total flooded area (sum of 5km cells): 0.1494 km2
    Total flooded volume (sum of 5km cells): 40271.40 m3
[86/154] Processing res_105_2071_86_Ens01_binary_10cm.tif (event=86)
Skippping QA
    Total flooded area (sum of 5km cells): 1.8927 km2
    Total flooded volume (sum of 5km cells): 991856.69 m3
[87/154] Processing res_105_2072_87_Ens01_binary_10cm.tif (event=87)
Skippping QA
    Total flooded area (sum of 5km cells): 0.6201 km2
    Total flooded volume (sum of 5km cells): 214901.09 m3
[88/154] Processing res_105_2072_88_Ens01_binary_10cm.tif (event=88)
Skippping QA
    Total flooded area (sum of

    Total flooded area (sum of 5km cells): 2.8485 km2
    Total flooded volume (sum of 5km cells): 879522.38 m3
[126/154] Processing res_105_2077_126_Ens01_binary_10cm.tif (event=126)
Skippping QA
    Total flooded area (sum of 5km cells): 3.7260 km2
    Total flooded volume (sum of 5km cells): 1368970.25 m3
[127/154] Processing res_105_2077_127_Ens01_binary_10cm.tif (event=127)
Skippping QA
    Total flooded area (sum of 5km cells): 1.7190 km2
    Total flooded volume (sum of 5km cells): 648028.81 m3
[128/154] Processing res_105_2077_128_Ens01_binary_10cm.tif (event=128)
Skippping QA
    Total flooded area (sum of 5km cells): 2.3166 km2
    Total flooded volume (sum of 5km cells): 802943.19 m3
[129/154] Processing res_105_2077_129_Ens01_binary_10cm.tif (event=129)
Skippping QA
    Total flooded area (sum of 5km cells): 0.4266 km2
    Total flooded volume (sum of 5km cells): 132009.31 m3
[130/154] Processing res_105_2077_130_Ens01_binary_10cm.tif (event=130)
Skippping QA
    Total floo

    Total flooded area (sum of 5km cells): 0.2493 km2
    Total flooded volume (sum of 5km cells): 154284.31 m3
[11/154] Processing res_105_2019_11_Ens01_binary_30cm.tif (event=11)
Skippping QA
    Total flooded area (sum of 5km cells): 0.3114 km2
    Total flooded volume (sum of 5km cells): 208304.09 m3
[12/154] Processing res_105_2020_12_Ens01_binary_30cm.tif (event=12)
Skippping QA
    Total flooded area (sum of 5km cells): 0.0189 km2
    Total flooded volume (sum of 5km cells): 9899.10 m3
[13/154] Processing res_105_2022_13_Ens01_binary_30cm.tif (event=13)
Skippping QA
    Total flooded area (sum of 5km cells): 0.1377 km2
    Total flooded volume (sum of 5km cells): 70163.10 m3
[14/154] Processing res_105_2022_14_Ens01_binary_30cm.tif (event=14)
Skippping QA
    Total flooded area (sum of 5km cells): 1.1214 km2
    Total flooded volume (sum of 5km cells): 861607.75 m3
[15/154] Processing res_105_2023_15_Ens01_binary_30cm.tif (event=15)
Skippping QA
    Total flooded area (sum of 5k

...

    Total flooded area (sum of 5km cells): 9.0117 km2
    Total flooded volume (sum of 5km cells): 10218342.00 m3
[245/409] Processing res_105_2062_245_Ens04_binary_30cm.tif (event=245)
Skippping QA
    Total flooded area (sum of 5km cells): 0.6048 km2
    Total flooded volume (sum of 5km cells): 496116.03 m3
[246/409] Processing res_105_2062_246_Ens04_binary_30cm.tif (event=246)
Skippping QA
    Total flooded area (sum of 5km cells): 1.2222 km2
    Total flooded volume (sum of 5km cells): 866810.75 m3
[247/409] Processing res_105_2062_247_Ens04_binary_30cm.tif (event=247)
Skippping QA
    Total flooded area (sum of 5km cells): 3.1185 km2
    Total flooded volume (sum of 5km cells): 2292398.25 m3
[248/409] Processing res_105_2062_248_Ens04_binary_30cm.tif (event=248)
Skippping QA
    Total flooded area (sum of 5km cells): 5.3415 km2
    Total flooded volume (sum of 5km cells): 4890568.00 m3
[249/409] Processing res_105_2062_249_Ens04_binary_30cm.tif (event=249)
Skippping QA
    Total f

    Total flooded area (sum of 5km cells): 1.1871 km2
    Total flooded volume (sum of 5km cells): 875007.00 m3
[287/409] Processing res_105_2067_287_Ens04_binary_30cm.tif (event=287)
Skippping QA
    Total flooded area (sum of 5km cells): 1.5102 km2
    Total flooded volume (sum of 5km cells): 1164268.75 m3
[288/409] Processing res_105_2067_288_Ens04_binary_30cm.tif (event=288)
Skippping QA
    Total flooded area (sum of 5km cells): 5.4189 km2
    Total flooded volume (sum of 5km cells): 4990269.00 m3
[289/409] Processing res_105_2067_289_Ens04_binary_30cm.tif (event=289)
Skippping QA
    Total flooded area (sum of 5km cells): 0.9180 km2
    Total flooded volume (sum of 5km cells): 676429.25 m3
[290/409] Processing res_105_2067_290_Ens04_binary_30cm.tif (event=290)
Skippping QA
    Total flooded area (sum of 5km cells): 0.2142 km2
    Total flooded volume (sum of 5km cells): 141743.70 m3
[291/409] Processing res_105_2067_291_Ens04_binary_30cm.tif (event=291)
Skippping QA
    Total flo

    Total flooded area (sum of 5km cells): 3.1410 km2
    Total flooded volume (sum of 5km cells): 2423744.00 m3
[329/409] Processing res_105_2072_329_Ens04_binary_30cm.tif (event=329)
Skippping QA
    Total flooded area (sum of 5km cells): 2.5317 km2
    Total flooded volume (sum of 5km cells): 1357346.75 m3
[330/409] Processing res_105_2072_330_Ens04_binary_30cm.tif (event=330)
Skippping QA
    Total flooded area (sum of 5km cells): 0.4959 km2
    Total flooded volume (sum of 5km cells): 266398.19 m3
[331/409] Processing res_105_2072_331_Ens04_binary_30cm.tif (event=331)
Skippping QA
    Total flooded area (sum of 5km cells): 0.8091 km2
    Total flooded volume (sum of 5km cells): 380232.91 m3
[332/409] Processing res_105_2073_332_Ens04_binary_30cm.tif (event=332)
Skippping QA
    Total flooded area (sum of 5km cells): 0.0594 km2
    Total flooded volume (sum of 5km cells): 31726.80 m3
[333/409] Processing res_105_2073_333_Ens04_binary_30cm.tif (event=333)
Skippping QA
    Total floo

    Total flooded area (sum of 5km cells): 0.5373 km2
    Total flooded volume (sum of 5km cells): 319494.62 m3
[371/409] Processing res_105_2077_371_Ens04_binary_30cm.tif (event=371)
Skippping QA
    Total flooded area (sum of 5km cells): 0.4878 km2
    Total flooded volume (sum of 5km cells): 285601.50 m3
[372/409] Processing res_105_2077_372_Ens04_binary_30cm.tif (event=372)
Skippping QA
    Total flooded area (sum of 5km cells): 0.0423 km2
    Total flooded volume (sum of 5km cells): 22226.40 m3
[373/409] Processing res_105_2077_373_Ens04_binary_30cm.tif (event=373)
Skippping QA
    Total flooded area (sum of 5km cells): 0.0153 km2
    Total flooded volume (sum of 5km cells): 8184.60 m3
[374/409] Processing res_105_2077_374_Ens04_binary_30cm.tif (event=374)
Skippping QA
    Total flooded area (sum of 5km cells): 1.0269 km2
    Total flooded volume (sum of 5km cells): 685070.12 m3
[375/409] Processing res_105_2077_375_Ens04_binary_30cm.tif (event=375)
Skippping QA
    Total flooded 

Done! Saved 409 events to /scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/5km_total/Catchment_105/Ens04_105/30cm/flooded_volume_5km_total_Ens04_105_30cm.nc

Processing Ens05_105: 770 total events

[Ens05_105 | 10cm] Processing 385 events
[1/385] Processing res_105_1996_1_Ens05_binary_10cm.tif (event=1)
Skippping QA
    Total flooded area (sum of 5km cells): 0.1044 km2
    Total flooded volume (sum of 5km cells): 34537.50 m3
[2/385] Processing res_105_1999_2_Ens05_binary_10cm.tif (event=2)
Skippping QA
    Total flooded area (sum of 5km cells): 0.1548 km2
    Total flooded volume (sum of 5km cells): 50035.50 m3
[3/385] Processing res_105_1999_3_Ens05_binary_10cm.tif (event=3)
Skippping QA
    Total flooded area (sum of 5km cells): 0.6219 km2
    Total flooded volume (sum of 5km cells): 194212.80 m3
[4/385] Processing res_105_2001_4_Ens05_binary_10cm.tif (event=4)
Skippping QA
    Total flooded area (sum of 5km cells): 0.3510 km2
    Total flooded volume (sum of 5km cells):

In [ ]:
# if __name__ == "__main__":
#     parser = argparse.ArgumentParser(description="Aggregate 30m flood rasters to 5km totals")
#     parser.add_argument(
#         "ha_num",
#         nargs="?",
#         default="23",
#         help="Catchment number (e.g. 23)"
#     )
#     args = parser.parse_args()